# Загрузка данных и первичный анализ

In [14]:
import pandas as pd

df1 = pd.read_csv('data/S07-hw-dataset-01.csv')
sample_id_1 = df1['sample_id']
X1 = df1.drop(columns=['sample_id'])
df2 = pd.read_csv('data/S07-hw-dataset-02.csv')
sample_id_2 = df2['sample_id']
X2 = df2.drop(columns=['sample_id'])
df3 = pd.read_csv('data/S07-hw-dataset-03.csv')
sample_id_3 = df3['sample_id']
X3 = df3.drop(columns=['sample_id'])

print(df1.head())
print(df1.info())
print(df1.describe())
print("Пропуски:\n", df1.isnull().sum())

print(df2.head())
print(df2.info())
print(df2.describe())
print("Пропуски:\n", df2.isnull().sum())

print(df3.head())
print(df3.info())
print(df3.describe())
print("Пропуски:\n", df3.isnull().sum())

   sample_id        f01        f02       f03         f04        f05  \
0          0  -0.536647 -69.812900 -0.002657   71.743147 -11.396498   
1          1  15.230731  52.727216 -1.273634 -104.123302  11.589643   
2          2  18.542693  77.317150 -1.321686 -111.946636  10.254346   
3          3 -12.538905 -41.709458  0.146474   16.322124   1.391137   
4          4  -6.903056  61.833444 -0.022466  -42.631335   3.107154   

         f06        f07       f08  
0 -12.291287  -6.836847 -0.504094  
1  34.316967 -49.468873  0.390356  
2  25.892951  44.595250  0.325893  
3   2.014316 -39.930582  0.139297  
4  -5.471054   7.001149  0.131213  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  12000 non-null  int64  
 1   f01        12000 non-null  float64
 2   f02        12000 non-null  float64
 3   f03        12000 non-null  float64
 4   f04 

# Препроцессинг

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X1_scaled = scaler.fit_transform(X1)
X2_scaled = scaler.fit_transform(X2)
X3_scaled = scaler.fit_transform(X3)

# Модели недели 7

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
from itertools import product

def compute_metrics(X, labels, method_name=""):
    noise_mask = (labels == -1)
    noise_ratio = noise_mask.mean()
    if noise_ratio > 0:
        # Убираем шум перед расчётом метрик.
        X_clean = X[~noise_mask]
        labels_clean = labels[~noise_mask]
        if len(np.unique(labels_clean)) < 2:
            # Недостаточно кластеров после удаления шума.
            sil, db, ch = -1, np.inf, -1
        else:
            sil = silhouette_score(X_clean, labels_clean)
            db = davies_bouldin_score(X_clean, labels_clean)
            ch = calinski_harabasz_score(X_clean, labels_clean)
    else:
        sil = silhouette_score(X, labels)
        db = davies_bouldin_score(X, labels)
        ch = calinski_harabasz_score(X, labels)
    return {
        'silhouette': sil,
        'davies_bouldin': db,
        'calinski_harabasz': ch,
        'noise_ratio': noise_ratio
    }

print("=== Dataset-01 ===")

# KMeans.
k_range = range(2, 16)
sil_scores_1 = []
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X1_scaled)
    sil = silhouette_score(X1_scaled, labels)
    sil_scores_1.append(sil)

best_k1 = k_range[np.argmax(sil_scores_1)]
plt.figure(figsize=(6,4))
plt.plot(k_range, sil_scores_1, marker='o')
plt.title('Dataset-01: Silhouette vs k (KMeans)')
plt.xlabel('k'); plt.ylabel('Silhouette Score')
plt.savefig('artifacts/figures/sil_vs_k_ds1.png')
plt.close()

kmeans1 = KMeans(n_clusters=best_k1, random_state=42, n_init=10)
labels_kmeans1 = kmeans1.fit_predict(X1_scaled)
metrics_kmeans1 = compute_metrics(X1_scaled, labels_kmeans1)

# Agglomerative.
linkages = ['ward', 'complete', 'average']
agg_results_1 = {}
for linkage in linkages:
    if linkage == 'ward':
        agg = AgglomerativeClustering(n_clusters=best_k1, linkage=linkage)
    else:
        agg = AgglomerativeClustering(n_clusters=best_k1, linkage=linkage, metric='euclidean')
    labels_agg = agg.fit_predict(X1_scaled)
    metrics = compute_metrics(X1_scaled, labels_agg)
    agg_results_1[linkage] = {
        'labels': labels_agg,
        'metrics': metrics
    }

# Выберем лучший linkage по silhouette.
best_linkage1 = max(agg_results_1.keys(), key=lambda l: agg_results_1[l]['metrics']['silhouette'])
labels_agg1 = agg_results_1[best_linkage1]['labels']
metrics_agg1 = agg_results_1[best_linkage1]['metrics']

print(f"Dataset-01: KMeans k={best_k1}, Agg linkage='{best_linkage1}'")

# Dataset-02: KMeans + DBSCAN.
print("=== Dataset-02 ===")

# KMeans.
sil_scores_2 = []
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X2_scaled)
    sil = silhouette_score(X2_scaled, labels)
    sil_scores_2.append(sil)

best_k2 = k_range[np.argmax(sil_scores_2)]
plt.figure(figsize=(6,4))
plt.plot(k_range, sil_scores_2, marker='o')
plt.title('Dataset-02: Silhouette vs k (KMeans)')
plt.xlabel('k'); plt.ylabel('Silhouette Score')
plt.savefig('artifacts/figures/sil_vs_k_ds2.png')
plt.close()

kmeans2 = KMeans(n_clusters=best_k2, random_state=42, n_init=10)
labels_kmeans2 = kmeans2.fit_predict(X2_scaled)
metrics_kmeans2 = compute_metrics(X2_scaled, labels_kmeans2)

# DBSCAN.
eps_range = np.arange(0.1, 2.1, 0.1)
min_samples_range = range(3, 11)
dbscan_results_2 = {}

for eps, min_samples in product(eps_range, min_samples_range):
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X2_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters < 2:
        continue
    metrics = compute_metrics(X2_scaled, labels)
    key = (eps, min_samples)
    dbscan_results_2[key] = {
        'labels': labels,
        'metrics': metrics
    }

# Выбираем лучший DBSCAN по silhouette (но с разумным шумом, например < 30%).
valid_dbscan = {
    k: v for k, v in dbscan_results_2.items()
    if v['metrics']['noise_ratio'] < 0.3 and v['metrics']['silhouette'] > 0
}
if valid_dbscan:
    best_key2 = max(valid_dbscan.keys(), key=lambda k: valid_dbscan[k]['metrics']['silhouette'])
    labels_dbscan2 = valid_dbscan[best_key2]['labels']
    metrics_dbscan2 = valid_dbscan[best_key2]['metrics']
    best_eps2, best_ms2 = best_key2
else:
    # fallback
    best_key2 = list(dbscan_results_2.keys())[0]
    labels_dbscan2 = dbscan_results_2[best_key2]['labels']
    metrics_dbscan2 = dbscan_results_2[best_key2]['metrics']
    best_eps2, best_ms2 = best_key2

print(f"Dataset-02: KMeans k={best_k2}, DBSCAN eps={best_eps2:.2f}, min_samples={best_ms2}")

# .Dataset-03: KMeans + DBSCAN.
print("=== Dataset-03 ===")

# KMeans.
sil_scores_3 = []
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X3_scaled)
    sil = silhouette_score(X3_scaled, labels)
    sil_scores_3.append(sil)

best_k3 = k_range[np.argmax(sil_scores_3)]
plt.figure(figsize=(6,4))
plt.plot(k_range, sil_scores_3, marker='o')
plt.title('Dataset-03: Silhouette vs k (KMeans)')
plt.xlabel('k'); plt.ylabel('Silhouette Score')
plt.savefig('artifacts/figures/sil_vs_k_ds3.png')
plt.close()

kmeans3 = KMeans(n_clusters=best_k3, random_state=42, n_init=10)
labels_kmeans3 = kmeans3.fit_predict(X3_scaled)
metrics_kmeans3 = compute_metrics(X3_scaled, labels_kmeans3)

# DBSCAN.
dbscan_results_3 = {}
for eps, min_samples in product(eps_range, min_samples_range):
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X3_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters < 2:
        continue
    metrics = compute_metrics(X3_scaled, labels)
    key = (eps, min_samples)
    dbscan_results_3[key] = {
        'labels': labels,
        'metrics': metrics
    }

valid_dbscan3 = {
    k: v for k, v in dbscan_results_3.items()
    if v['metrics']['noise_ratio'] < 0.3 and v['metrics']['silhouette'] > 0
}
if valid_dbscan3:
    best_key3 = max(valid_dbscan3.keys(), key=lambda k: valid_dbscan3[k]['metrics']['silhouette'])
    labels_dbscan3 = valid_dbscan3[best_key3]['labels']
    metrics_dbscan3 = valid_dbscan3[best_key3]['metrics']
    best_eps3, best_ms3 = best_key3
else:
    best_key3 = list(dbscan_results_3.keys())[0]
    labels_dbscan3 = dbscan_results_3[best_key3]['labels']
    metrics_dbscan3 = dbscan_results_3[best_key3]['metrics']
    best_eps3, best_ms3 = best_key3

print(f"Dataset-03: KMeans k={best_k3}, DBSCAN eps={best_eps3:.2f}, min_samples={best_ms3}")

=== Dataset-01 ===
Dataset-01: KMeans k=2, Agg linkage='ward'
=== Dataset-02 ===
Dataset-02: KMeans k=2, DBSCAN eps=0.80, min_samples=10
=== Dataset-03 ===
Dataset-03: KMeans k=3, DBSCAN eps=0.80, min_samples=3


# Метрики качества

In [17]:
import json

# Соберём все метрики.
metrics_summary = {
    "dataset-01": {
        "KMeans": {
            "silhouette_score": float(metrics_kmeans1['silhouette']),
            "davies_bouldin_score": float(metrics_kmeans1['davies_bouldin']),
            "calinski_harabasz_score": float(metrics_kmeans1['calinski_harabasz']),
            "noise_ratio": 0.0
        },
        f"Agglomerative ({best_linkage1})": {
            "silhouette_score": float(metrics_agg1['silhouette']),
            "davies_bouldin_score": float(metrics_agg1['davies_bouldin']),
            "calinski_harabasz_score": float(metrics_agg1['calinski_harabasz']),
            "noise_ratio": 0.0
        }
    },
    "dataset-02": {
        "KMeans": {
            "silhouette_score": float(metrics_kmeans2['silhouette']),
            "davies_bouldin_score": float(metrics_kmeans2['davies_bouldin']),
            "calinski_harabasz_score": float(metrics_kmeans2['calinski_harabasz']),
            "noise_ratio": 0.0
        },
        f"DBSCAN (eps={best_eps2:.2f}, min_samples={best_ms2})": {
            "silhouette_score": float(metrics_dbscan2['silhouette']),
            "davies_bouldin_score": float(metrics_dbscan2['davies_bouldin']),
            "calinski_harabasz_score": float(metrics_dbscan2['calinski_harabasz']),
            "noise_ratio": float(metrics_dbscan2['noise_ratio'])
        }
    },
    "dataset-03": {
        "KMeans": {
            "silhouette_score": float(metrics_kmeans3['silhouette']),
            "davies_bouldin_score": float(metrics_kmeans3['davies_bouldin']),
            "calinski_harabasz_score": float(metrics_kmeans3['calinski_harabasz']),
            "noise_ratio": 0.0
        },
        f"DBSCAN (eps={best_eps3:.2f}, min_samples={best_ms3})": {
            "silhouette_score": float(metrics_dbscan3['silhouette']),
            "davies_bouldin_score": float(metrics_dbscan3['davies_bouldin']),
            "calinski_harabasz_score": float(metrics_dbscan3['calinski_harabasz']),
            "noise_ratio": float(metrics_dbscan3['noise_ratio'])
        }
    }
}

with open('artifacts/metrics_summary.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_summary, f, indent=4, ensure_ascii=False)

print("Доля шума в DBSCAN")
print(f"Dataset-02: {metrics_dbscan2['noise_ratio']:.2%}")
print(f"Dataset-03: {metrics_dbscan3['noise_ratio']:.2%}")


Доля шума в DBSCAN
Dataset-02: 1.18%
Dataset-03: 0.15%


# Визуализация

In [18]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Dataset-01: выбрать лучшую модель.
if metrics_kmeans1['silhouette'] >= metrics_agg1['silhouette']:
    best_labels_1 = labels_kmeans1
    best_title_1 = f"Dataset-01: KMeans (k={best_k1})"
else:
    best_labels_1 = labels_agg1
    best_title_1 = f"Dataset-01: Agglomerative ({best_linkage1})"

# PCA + scatter.
pca = PCA(n_components=2, random_state=42)
X1_pca = pca.fit_transform(X1_scaled)
plt.figure(figsize=(6, 5))
plt.scatter(X1_pca[:, 0], X1_pca[:, 1], c=best_labels_1, cmap='tab10', s=15)
plt.title(best_title_1)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.tight_layout()
plt.savefig('artifacts/figures/pca_ds1.png')
plt.close()

# Dataset-02.
if metrics_kmeans2['silhouette'] >= metrics_dbscan2['silhouette']:
    best_labels_2 = labels_kmeans2
    best_title_2 = f"Dataset-02: KMeans (k={best_k2})"
else:
    best_labels_2 = labels_dbscan2
    best_title_2 = f"Dataset-02: DBSCAN (eps={best_eps2:.2f}, ms={best_ms2})"
    
X2_pca = pca.fit_transform(X2_scaled)
plt.figure(figsize=(6, 5))
plt.scatter(X2_pca[:, 0], X2_pca[:, 1], c=best_labels_2, cmap='tab10', s=15)
plt.title(best_title_2)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.tight_layout()
plt.savefig('artifacts/figures/pca_ds2.png')
plt.close()

# Dataset-03.
if metrics_kmeans3['silhouette'] >= metrics_dbscan3['silhouette']:
    best_labels_3 = labels_kmeans3
    best_title_3 = f"Dataset-03: KMeans (k={best_k3})"
else:
    best_labels_3 = labels_dbscan3
    best_title_3 = f"Dataset-03: DBSCAN (eps={best_eps3:.2f}, ms={best_ms3})"

X3_pca = pca.fit_transform(X3_scaled)
plt.figure(figsize=(6, 5))
plt.scatter(X3_pca[:, 0], X3_pca[:, 1], c=best_labels_3, cmap='tab10', s=15)
plt.title(best_title_3)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.tight_layout()
plt.savefig('artifacts/figures/pca_ds3.png')
plt.close()

# Устойчивость

In [19]:
from sklearn.metrics import adjusted_rand_score
import numpy as np

# Используем тот же k, что и ранее.
k_stable = best_k1
n_runs = 5
all_labels = []

# Запускаем KMeans 5 раз с разными random_state.
for i in range(n_runs):
    kmeans_run = KMeans(n_clusters=k_stable, random_state=i, n_init=10)
    labels_run = kmeans_run.fit_predict(X1_scaled)
    all_labels.append(labels_run)

# Считаем попарные ARI между всеми запусками.
ari_matrix = np.zeros((n_runs, n_runs))
for i in range(n_runs):
    for j in range(i + 1, n_runs):
        ari = adjusted_rand_score(all_labels[i], all_labels[j])
        ari_matrix[i, j] = ari
        ari_matrix[j, i] = ari

# Средний ARI по всем парам.
pairwise_aris = ari_matrix[np.triu_indices(n_runs, k=1)]
mean_ari = pairwise_aris.mean()

print(f"Попарные ARI между {n_runs} запусками KMeans (k={k_stable}):")
print(np.round(pairwise_aris, 4))
print(f"\nСредний ARI: {mean_ari:.4f}")

# Интерпретация.
if mean_ari > 0.95:
    stability_comment = "Разбиения практически идентичны, значит модель устойчива."
elif mean_ari > 0.85:
    stability_comment = "Высокая устойчивость: разбиения близки."
else:
    stability_comment = "Низкая устойчивость: результаты сильно зависят от инициализации."

print(f"\nВывод: {stability_comment}")

Попарные ARI между 5 запусками KMeans (k=2):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

Средний ARI: 1.0000

Вывод: Разбиения практически идентичны, значит модель устойчива.


## Итог по каждому датасету

### Dataset-01
Лучшим методом признан **KMeans с k=2** (silhouette ≈ 0.522), несмотря на то, что Agglomerative с linkage='ward' показал близкий результат. Датасет состоит из числовых признаков в сильно разных масштабах, но после StandardScaler структура стала компактной и выпуклой — идеально для KMeans. Выбросов и шума нет, поэтому плотностные методы избыточны. KMeans дал стабильное, интерпретируемое разбиение без шума.

### Dataset-02
Лучше всего справился **DBSCAN (eps=0.80, min_samples=10)** (silhouette ≈ 0.414), тогда как KMeans (k=2) получил низкий silhouette (≈ 0.307). Это связано с нелинейной формой кластеров и наличием выбросов: KMeans пытался разделить данные на "шары", что исказило структуру. DBSCAN корректно выделил плотные регионы и пометил выбросы как шум (~1.2%), что соответствует природе данных.

### Dataset-03
Выбран **DBSCAN (eps=0.80, min_samples=3)** (silhouette ≈ 0.373), поскольку кластеры имеют разную плотность, а фон содержит рассеянный шум. KMeans (k=3) дал формальные, но геометрически неверные границы. DBSCAN адаптировался к локальной плотности и выделил компактные группы, отнеся ~0.1% точек к шуму — это согласуется с замыслом датасета.

# Сохранение в labels/ и best_configs.json

In [20]:
import json
import pandas as pd


# Определяем лучшие метки для каждого датасета.
# Dataset-01.
if metrics_kmeans1['silhouette'] >= metrics_agg1['silhouette']:
    best_labels_1 = labels_kmeans1
    best_method_1 = "KMeans"
    best_params_1 = {"n_clusters": int(best_k1)}
else:
    best_labels_1 = labels_agg1
    best_method_1 = f"Agglomerative ({best_linkage1})"
    best_params_1 = {"linkage": best_linkage1, "n_clusters": int(best_k1)}

# Dataset-02.
if metrics_kmeans2['silhouette'] >= metrics_dbscan2['silhouette']:
    best_labels_2 = labels_kmeans2
    best_method_2 = "KMeans"
    best_params_2 = {"n_clusters": int(best_k2)}
else:
    best_labels_2 = labels_dbscan2
    best_method_2 = "DBSCAN"
    best_params_2 = {"eps": float(best_eps2), "min_samples": int(best_ms2)}

# Dataset-03.
if metrics_kmeans3['silhouette'] >= metrics_dbscan3['silhouette']:
    best_labels_3 = labels_kmeans3
    best_method_3 = "KMeans"
    best_params_3 = {"n_clusters": int(best_k3)}
else:
    best_labels_3 = labels_dbscan3
    best_method_3 = "DBSCAN"
    best_params_3 = {"eps": float(best_eps3), "min_samples": int(best_ms3)}

# Сохраняем CSV с метками.
pd.DataFrame({'sample_id': sample_id_1, 'cluster_label': best_labels_1}).to_csv('artifacts/labels/labels_hw07_ds1.csv', index=False)
pd.DataFrame({'sample_id': sample_id_2, 'cluster_label': best_labels_2}).to_csv('artifacts/labels/labels_hw07_ds2.csv', index=False)
pd.DataFrame({'sample_id': sample_id_3, 'cluster_label': best_labels_3}).to_csv('artifacts/labels/labels_hw07_ds3.csv', index=False)

# Сохраняем best_configs.json.
best_configs = {
    "dataset-01": {
        "method": best_method_1,
        "params": best_params_1,
        "criterion": "max silhouette_score"
    },
    "dataset-02": {
        "method": best_method_2,
        "params": best_params_2,
        "criterion": "max silhouette_score with noise_ratio < 0.3"
    },
    "dataset-03": {
        "method": best_method_3,
        "params": best_params_3,
        "criterion": "max silhouette_score with noise_ratio < 0.3"
    }
}

with open('artifacts/best_configs.json', 'w', encoding='utf-8') as f:
    json.dump(best_configs, f, indent=4, ensure_ascii=False)